# Strategic Identification of Emerging Olympic Sports for Brisbane 2032

## Project Overview

This project investigates how participation data and media visibility indicators can support strategic investment decisions ahead of the Brisbane 2032 Olympic Games.

The analysis began by examining overall participation patterns across Australian sports. Rather than focusing solely on the most popular sports, the project explores whether mid-tier Olympic sports with strong youth participation may represent better opportunities for targeted development and support.

The project combines:
- Exploratory analysis of AusPlay participation data
- Youth participation profiling
- Media visibility analysis using Guardian Australia articles
- Comparative evaluation of participation and coverage patterns

The goal is to identify sports that demonstrate participation potential but may receive comparatively limited public attention.


### Question

Which Olympic sports present the most promising opportunities for fan development among school-age children in Australia, based on current participation levels and year-on-year trends?

This question is relevant because the ASC's 2032 strategy relies on identifying sports where a meaningful pool of young Australians is already active. Children who already participate in a sport are a natural audience for building fan connections to the Olympic athletes competing in that same discipline. Rather than focusing on already-dominant sports that receive enormous funding and media attention, the aim is to find the sports sitting just below that level, where targeted ASC investment could make a genuine difference.

In [26]:
import pandas as pd

FILE = "../src/C4S-AusPlay-By-Sport-Data-Tables-13-November-2025.xlsx"

# Sheet 2 has two sections.
# Rows 15-156: participation RATES (proportion of child population) calculated by AusPlay.
# Rows 158+:   estimated participant COUNTS.
# We load both and merge them so each sport has both a count and a rate.

rates_raw = pd.read_excel(
    FILE,
    sheet_name='2',
    skiprows=14,
    nrows=142,
    header=None
)

# AusPlay's overall child participation rate is already provided
# in column 1. Age-specific rates are retained separately.
rates_raw = rates_raw.iloc[:, [0, 1, 7, 8, 9]]
rates_raw.columns = [
    "Activity",
    "YouthRate",
    "Rate_5_8",
    "Rate_9_11",
    "Rate_12_14"
]

In [25]:
counts_raw = pd.read_excel(
    FILE,
    sheet_name='2',
    skiprows=158,
    nrows=142,
    header=None
)

counts_raw = counts_raw.iloc[:, [0, 7, 8, 9]]
counts_raw.columns = [
    "Activity",
    "Count_5_8",
    "Count_9_11",
    "Count_12_14"
]

counts_raw["YouthCount"] = (
    counts_raw["Count_5_8"]
    + counts_raw["Count_9_11"]
    + counts_raw["Count_12_14"]
)

ausplay_data = counts_raw[["Activity", "YouthCount"]].merge(
    rates_raw[["Activity", "YouthRate"]],
    on="Activity"
)

ausplay_data.head()

,Activity,YouthCount,YouthRate
0,Adventure racing,7503.477723,0.002239
1,Air sports,8342.603488,0.002589
2,Archery,5379.755633,0.001481
3,"Athletics, track and field",136658.053740,0.032402
4,Australian football,283390.828971,0.063026


### Data Preparation

Sheet 2 contains two sections of data for the same set of sports. The first section (rows 15 to 156) records participation rates that AusPlay has already calculated against the child population baseline. The second section (rows 158 onwards) records the estimated number of participants.

Rather than calculating a youth rate manually by dividing counts, the rates from AusPlay's own section are used directly. This is more reliable because AusPlay applies its own weighting and adjustment methodology to produce those figures.

The two sections are loaded separately and merged on the activity name. The age bands 5-8, 9-11 and 12-14 are combined into a single YouthCount (raw participants) and a single YouthRate (proportion of the child population). The 0-4 band is excluded throughout because toddlers are not the audience for a 2032 fan development strategy.

In [3]:
olympic_sports = [
    "Athletics, track and field", "Running/jogging",
    "Swimming", "Football/soccer", "Basketball", "Gymnastics", "Tennis",
    "Cycling", "Hockey", "Taekwondo/Taekwon-do",
    "Volleyball (indoor and outdoor)", "Badminton", "Surfing",
    "Table tennis", "Golf", "Boxing", "Equestrian",
    "Mountain biking", "BMX", "Sport climbing",
    "Judo", "Rowing", "Archery", "Handball", "Sailing",
    "Canoeing/Kayaking", "Fencing", "Weight lifting", "Triathlon", "Wrestling"
]

olympic_data = (
    ausplay_data[ausplay_data["Activity"].isin(olympic_sports)]
    .sort_values("YouthCount", ascending=False)
    .reset_index(drop=True)
)

olympic_data

,Activity,YouthCount,YouthRate
0,Swimming,799135.077742,0.235637
1,Football/soccer,618174.829863,0.145308
2,Basketball,326756.192889,0.070958
3,Gymnastics,242049.075452,0.07088
4,Tennis,180086.975223,0.039848
5,"Athletics, track and field",136658.053740,0.032402
6,Running/jogging,118233.788008,0.030488
7,Cycling,63435.695534,0.017208
8,Hockey,43593.396053,0.0096
9,Taekwondo/Taekwon-do,37591.960717,0.008274


### Filtering to Confirmed Olympic Sports

The AusPlay dataset covers a wide range of activities, many of which have no Olympic programme connection. The list above retains only sports with a confirmed or long-standing presence on the Olympic programme. A few deliberate exclusions are worth noting:

Rugby league and Touch football were excluded because neither is an Olympic sport. Rugby Sevens is, but it has a distinct playing format and separate participation pathway, and it does not appear as its own activity in the AusPlay data.

Karate was excluded because it appeared only at the Tokyo 2020 Games and was not included in Paris 2024 or the confirmed Brisbane 2032 programme.

The sports that remain are all either on the confirmed Olympic programme or have been present across multiple recent Games.

In [4]:
dominant_tier = olympic_data[olympic_data["YouthCount"] >= 100000]
mid_tier      = olympic_data[(olympic_data["YouthCount"] >= 10000) & (olympic_data["YouthCount"] < 100000)]
lower_tier    = olympic_data[olympic_data["YouthCount"] < 10000]

print(f"Dominant tier (100k+): {len(dominant_tier)} sports")
print(dominant_tier[["Activity", "YouthCount"]].to_string(index=False))
print(f"\nMid-tier (10k-100k): {len(mid_tier)} sports")
print(mid_tier[["Activity", "YouthCount"]].to_string(index=False))
print(f"\nLower tier (<10k): {len(lower_tier)} sports")

Dominant tier (100k+): 7 sports
                  Activity    YouthCount
                  Swimming 799135.077742
           Football/soccer 618174.829863
                Basketball 326756.192889
                Gymnastics 242049.075452
                    Tennis 180086.975223
Athletics, track and field 136658.053740
           Running/jogging 118233.788008

Mid-tier (10k-100k): 10 sports
                       Activity   YouthCount
                        Cycling 63435.695534
                         Hockey 43593.396053
           Taekwondo/Taekwon-do 37591.960717
Volleyball (indoor and outdoor) 33869.606334
                      Badminton 29070.477637
                        Surfing 22864.400898
                   Table tennis 20891.458566
                           Golf 20566.337381
                         Boxing 16368.819698
                     Equestrian 11183.373971

Lower tier (<10k): 13 sports


### Identifying the Mid-Tier

Ranking all Olympic sports by youth participation reveals three natural groupings.

The dominant tier (100,000 or more young participants) includes Swimming, Football/soccer, Basketball, Gymnastics, Tennis, Athletics and Running/jogging. These sports already receive the bulk of sporting media coverage, ASC funding and public attention. Investing further there would yield diminishing returns for fan development.

The lower tier (fewer than 10,000 young participants) covers sports like BMX, Sport climbing, Judo and Rowing. While these are genuine Olympic disciplines, their youth participation base is too small to build a meaningful fan community at scale before 2032.

The mid-tier sits between 10,000 and 100,000 young participants. These ten sports have substantial reach among school-age children but without the saturation of the dominant group. This is where targeted ASC engagement is most likely to produce a measurable impact, because the audience already exists but has not yet been fully connected to the Olympic programme.

In [5]:
import plotly.express as px

youth_chart = px.bar(
    mid_tier.sort_values("YouthCount"),
    x="YouthCount",
    y="Activity",
    orientation="h",
    title="Mid-Tier Olympic Sports: Youth Participation (ages 5-14), 2024-25",
    labels={"YouthCount": "Estimated Participants (ages 5-14)", "Activity": "Sport"},
    color="YouthRate",
    color_continuous_scale="Blues",
)

youth_chart.update_layout(coloraxis_colorbar=dict(title="Youth Rate"))
youth_chart.show()

### Visualisation

The bar length represents the absolute number of young participants (YouthCount), while the colour gradient represents the YouthRate - the proportion of the child population that participates in each sport, as calculated by AusPlay. Darker bars indicate a higher share of the child population.

Cycling is the largest sport in this group with around 63,000 young participants. Hockey has the smallest absolute count at roughly 44,000, but the colour indicates its YouthRate is not the lowest in the group, meaning it still captures a meaningful share of the child population relative to some other sports here.

Taken together, all ten sports have enough young participants to serve as a foundation for a genuine fan development programme.

In [6]:
yearly_changes = pd.read_excel(
    FILE, sheet_name='7', skiprows=15, nrows=142, usecols="A,B,D", header=None
)
yearly_changes.columns = ["Activity", "Rate_2023_24", "Rate_2024_25"]
yearly_changes["YearOnYear"] = yearly_changes["Rate_2024_25"] - yearly_changes["Rate_2023_24"]

sport_trends = yearly_changes[yearly_changes["Activity"].isin(mid_tier["Activity"])].copy()

sport_trends

,Activity,Rate_2023_24,Rate_2024_25,YearOnYear
5,Badminton,0.009746,0.007505,-0.002241
18,Boxing,0.003292,0.003632,0.000340
29,Cycling,0.028293,0.017208,-0.011085
37,Equestrian,0.004369,0.002948,-0.001422
48,Golf,0.007471,0.005073,-0.002398
54,Hockey,0.007600,0.009600,0.001999
119,Surfing,0.004505,0.004739,0.000234
123,Table tennis,0.004535,0.004450,-0.000085
124,Taekwondo/Taekwon-do,0.008179,0.008274,0.000095
133,Volleyball (indoor and outdoor),0.006241,0.007388,0.001147


In [7]:
sorted_trends = sport_trends.sort_values("YearOnYear")
bar_colours = ["#2ecc71" if v >= 0 else "#e74c3c" for v in sorted_trends["YearOnYear"]]

trend_chart = px.bar(
    sorted_trends,
    x="YearOnYear",
    y="Activity",
    orientation="h",
    title="Year-on-Year Change in Participation Rate (2023-24 to 2024-25)",
    labels={"YearOnYear": "Change in Participation Rate", "Activity": "Sport"}
)

trend_chart.update_traces(marker_color=bar_colours)
trend_chart.add_vline(x=0, line_width=1.5, line_dash="dash", line_color="grey")
trend_chart.show()

### Trend Analysis

Sheet 7 records each sport's participation rate for two consecutive years (2023-24 and 2024-25). Using rates rather than raw counts controls for year-to-year changes in the child population, making the comparison more meaningful.

Green bars indicate growth in participation share, red bars indicate a decline. The dashed line at zero makes the direction immediately visible without needing to read individual values.

### Insights

Five of the ten mid-tier sports recorded a positive year-on-year trend: Hockey, Volleyball, Boxing, Surfing and Taekwondo. The other five declined, with Cycling showing the steepest fall.

**Hockey** is the strongest candidate for ASC focus. It combines a growing participation trend (+0.002) with a youth participation rate that reflects a genuinely youth-dominated sport. A direct peer connection programme between junior club players and the Kookaburras or Hockeyroos would directly address suggestion 1 from the scenario, since many current junior players could realistically be competing in 2032.

**Volleyball** is the second most compelling finding. With around 34,000 young participants and a positive trend (+0.001), it sits in a strong position. Volleyball also benefits from being played in school PE programmes, which gives the ASC an accessible entry point for engagement that does not depend on formal club membership.

**Surfing and Taekwondo** are both growing modestly. Surfing in particular has strong cultural resonance in Australia and has been on the Olympic programme since Tokyo 2020, yet its youth participation of around 23,000 suggests it has not yet fully capitalised on that Olympic profile. A targeted campaign connecting junior surfers to the Australian Olympic surf team could help convert that existing participation into active fan interest.

**Cycling** is the most complex case. It has the largest youth count in the mid-tier (63,000) but the steepest participation decline (-0.011). This decline likely reflects the shift of casual cycling away from organised participation rather than a loss of interest in the activity itself. The ASC could explore whether connecting young riders to the breadth of Olympic cycling disciplines (road, track, BMX, mountain bike) might re-engage a cohort that is active but not currently captured in participation data.

**Limitation:** The YouthRate values come from AusPlay's survey-based estimates and carry a margin of error, particularly for smaller sports. Year-on-year changes for sports like Taekwondo (+0.000095) and Table tennis (-0.000085) are effectively within the survey's noise threshold and should not be over-interpreted.

Across all three suggestions in the scenario: the mid-tier sports identified here already have active youth communities (suggestion 2), several have participants young enough to be Olympic contenders by 2032 (suggestion 1), and the 12-14 age band in particular will be entering peak social media use in the next few years (suggestion 3), making this group the most strategic audience for the ASC to engage now.

#### End of Phase 1
---
---

## Phase 2: Media Visibility Analysis

### Objective

Participation data alone does not indicate whether a sport receives public attention. To better understand the visibility of emerging Olympic sports, media coverage was analysed using articles retrieved from the Guardian API.

### Research Question

Do sports with growing youth participation receive comparable levels of media attention, or are some potentially underrepresented despite strong participation trends?

### Data Source

Articles were collected from the Guardian API using sport-specific search terms relevant to the candidate sports identified during the exploratory analysis.

### Method

Text data was cleaned and processed using TF-IDF vectorisation to identify dominant themes and compare relative media visibility across sports.

### Purpose

The results provide an additional perspective beyond participation rates, helping identify sports that may have growth potential but limited public exposure.

### Question

Which Olympic sports receive the most media attention in Guardian Australia's coverage of Brisbane 2032, and how does that compare to the mid-tier sports identified in Phase 1 as having the strongest youth participation?

This question matters because participation and media coverage are two separate things, and both are needed to convert a young participant into a fan. A child who plays hockey on weekends will only develop a connection to Olympic hockey if they also encounter it in the media around them. If the sports with the strongest youth participation from Phase 1 are receiving little to no coverage in the lead-up to 2032, that is a concrete and actionable gap for the ASC. This connects directly to suggestion 3 in the scenario - early adolescents who are building their media habits now will be adults by 2032, and the sports they read and hear about today will shape what they choose to follow.

In [8]:
import requests
import json
import re
import time
import pandas as pd

`requests` handles the API call, `json` parses the response, `re` strips HTML tags from article bodies, and `time` adds a delay between page requests to avoid hitting the API rate limit.

In [9]:
with open('../private/guardian_key.txt', 'r') as file:
    api_key = file.read().strip()
len(api_key)

36

The API key is loaded from a local file rather than being hard-coded in the notebook. This follows standard data engineering and security practice by keeping credentials separate from analytical code and reducing the risk of accidental disclosure.

### Data

I queried the Guardian API filtering to Australian-produced articles only using `production-office=aus`. This is important because the goal is to understand the media environment that young Australians are actually exposed to, not international or UK coverage of the Games.

The search uses 'Brisbane 2032 AND Olympic AND youth' to target articles specifically about the upcoming Games and their connection to young people. Without the 'youth' term the search returns thousands of results covering Brisbane city news, AFL coverage, and general articles that mention the Olympics only in passing. Adding 'youth' narrows the focus to content more relevant to the scenario.

The start date of January 2024 was chosen to capture the growing lead-up conversation as awareness of 2032 builds, while avoiding earlier articles where mentions were sparse and mostly infrastructural. I capped the fetch at 5 pages (50 articles) rather than pulling all 285 available pages. This is a deliberate decision - 50 articles is sufficient for text-based frequency analysis, avoids unnecessary API usage, and keeps the analysis reproducible without long wait times.

In [10]:
base_url = 'https://content.guardianapis.com/'
query_terms = "Brisbane 2032 AND Olympic AND youth"
office_filter = "aus"
start_date = "2024-01-01"

query_url = (base_url +
             f"search?q={query_terms}" +
             f"&production-office={office_filter}" +
             f"&from-date={start_date}" +
             f"&show-fields=body" +
             f"&api-key={api_key}")

print(query_url[:120])

https://content.guardianapis.com/search?q=Brisbane 2032 AND Olympic AND youth&production-office=aus&from-date=2024-01-01


The API request URL is assembled programmatically using the selected search parameters. Displaying a shortened preview of the URL provides a simple validation check that the query has been constructed correctly while keeping credentials hidden.

In [11]:
api_response = requests.get(query_url)
raw_json = api_response.json()
response_body = raw_json.get('response', '')

if response_body == '':
    print("ERROR:", raw_json)
else:
    print("SUCCESS!")
    print(f"{response_body['total']} results found in {response_body['pages']} pages")
    print(f"{response_body['pageSize']} results per page")
    page_results = response_body.get('results', [])

SUCCESS!
3363 results found in 337 pages
10 results per page


I always check the total result count before fetching everything. With 2,844 results across 285 pages, running the full fetch would take over 7 minutes and exhaust API call limits. Seeing this count first is what informed the decision to cap at 5 pages.

In [12]:
def extract_articles(page_results):
    article_dict = {}
    for item in page_results:
        pub_date = item['webPublicationDate']
        headline = item['webTitle'] + f" [{pub_date}]"
        raw_html = item['fields']['body']
        clean_text = re.sub(r'<.*?>', '', raw_html)
        article_dict[headline] = clean_text
    return article_dict


def fetch_pages(response_json, query_url, max_pages=5):
    page_limit = min(response_json['pages'], max_pages)
    print(f"Fetching from {page_limit} pages (of {response_json['pages']} available)...")

    collected = {}
    first_page = extract_articles(response_json['results'])
    collected.update(first_page)
    print("Added articles for page: 1")

    for pg in range(2, page_limit + 1):
        print("Getting articles from API for page:", pg)
        pg_response = requests.get(query_url + f"&page={pg}")
        pg_data = pg_response.json()['response']
        pg_articles = extract_articles(pg_data['results'])
        collected.update(pg_articles)
        print(f"Added {len(pg_articles)} articles for page {pg}. Total so far: {len(collected)}")
        time.sleep(1)

    print(f"FINISHED: {len(collected)} articles fetched.")
    return collected

In [13]:
guardian_articles = fetch_pages(response_body, query_url, max_pages=5)

Fetching from 5 pages (of 337 available)...
Added articles for page: 1
Getting articles from API for page: 2
Added 10 articles for page 2. Total so far: 20
Getting articles from API for page: 3
Added 10 articles for page 3. Total so far: 30
Getting articles from API for page: 4
Added 10 articles for page 4. Total so far: 40
Getting articles from API for page: 5
Added 10 articles for page 5. Total so far: 50
FINISHED: 50 articles fetched.


Setting `max_pages=5` gives 50 articles, which is enough text for a meaningful frequency analysis.

In [14]:
print("Total Articles:", len(guardian_articles))
for headline in guardian_articles.keys():
    print(headline)

Total Articles: 50
Concept images of controversial 2032 Brisbane Olympics precinct revealed – as it happened [2026-08-02T03:11:48Z]
Sydney blow away shellshocked Brisbane to revive ‘torpedoed’ AFL season [2026-09-05T08:26:35Z]
Brisbane Lions defeat Adelaide Crows: AFL semi-final – as it happened [2026-09-12T12:46:28Z]
Lions rewarded for embracing risk as ‘good Brisbane’ keep AFL flag dream alive | Jonathan Horn [2026-09-13T15:00:06Z]
Warning over future of Olympic Games if Brisbane 2032 program is not modernised [2026-09-07T07:14:42Z]
Alleged rapist arrested after returning to Brisbane crime scene that night, police say [2026-08-21T03:14:19Z]
AFL finals: where the Brisbane v Adelaide semi-final will be won and lost | Martin Pegan [2026-09-09T15:00:04Z]
Brisbane child accused of school stabbing allegedly faced ‘racist’ abuse, court hears [2026-07-15T06:07:36Z]
Teenager left with life-threatening injuries after alleged stabbing at Brisbane school [2026-07-14T07:38:43Z]
Brisbane swelters 

Printing the headlines is a useful sanity check before doing any text analysis. Looking through the list, it becomes clear that many of the 50 articles are about the Brisbane Lions AFL team, Brisbane weather events, or general Queensland politics - they appear in the results because they mention 'Brisbane' and 'youth' but are not about the Olympic Games specifically. This informed the decision in the analysis step to use targeted sport keyword counting rather than generic word frequency.

In [15]:
file_path = "../src/"
file_name = "brisbane_2032_sport_articles.json"

with open(f"{file_path}{file_name}", 'w', encoding='utf-8') as fp:
    fp.write(json.dumps(guardian_articles))

print("Saved.")

Saved.


Saving to JSON immediately after fetching means the API never needs to be called again. All subsequent analysis reads from this local file, which preserves API call quota and ensures the results are reproducible.

In [16]:
with open(f"{file_path}{file_name}", 'r', encoding='utf-8') as fp:
    guardian_articles = json.load(fp)

print(f"Loaded {len(guardian_articles)} articles from file.")

Loaded 50 articles from file.


### Analysis

An initial exploratory approach used general word-frequency analysis across the article corpus after removing common stopwords. The resulting terms were dominated by generic news language and live-blog terminology, providing little insight into sport-specific media attention.

To better align the analysis with the research objective, the approach was refined to use targeted sport-name keyword matching. A predefined list of Olympic sports was searched across the article corpus and occurrence counts were calculated for each sport. This produced a more interpretable measure of media visibility and enabled direct comparison between participation levels and media coverage.

This refinement improved the relevance of the results because it focused on identifying which sports receive attention in news reporting rather than which words appear most frequently in articles.

In [17]:
from collections import Counter

corpus_text = " ".join(guardian_articles.values())

# count specific sport names rather than all words - generic frequency
# was dominated by live blog noise (says, updated, mins, remaining)
sport_keywords = [
    "football", "soccer", "basketball", "tennis",
    "swimming", "athletics", "cricket", "rugby",
    "hockey", "cycling"
]

keyword_mentions = {}
for term in sport_keywords:
    keyword_mentions[term] = corpus_text.lower().count(term)

mentions_df = pd.DataFrame(list(keyword_mentions.items()), columns=["Sport", "Mentions"])
mentions_df = mentions_df.sort_values("Mentions", ascending=False)
mentions_df

,Sport,Mentions
0,football,22
6,cricket,12
5,athletics,10
1,soccer,3
7,rugby,2
4,swimming,2
9,cycling,2
3,tennis,1
2,basketball,1
8,hockey,0


The keyword list includes sports from both the dominant and mid-tier groups identified in Phase 1. This makes it possible to directly compare media coverage against participation data - for example, checking whether Hockey and Cycling (strong in Phase 1) appear as frequently as Football and Rugby (which dominate Australian media generally).

### Visualisation

Two charts are used. The first shows sport-specific mention counts and directly answers the question of which Olympic sports are present in the coverage. The second is a timeline showing article counts by month, which reveals whether coverage is growing as 2032 approaches or is being driven by unrelated events.

In [18]:
import plotly.express as px
import pandas as pd

mentions_chart = px.bar(
    mentions_df.sort_values("Mentions"),
    x="Mentions",
    y="Sport",
    orientation="h",
    title="Sports Mentioned in Guardian Australia's Brisbane 2032 Coverage",
    labels={"Mentions": "Number of Mentions", "Sport": "Sport"}
)
mentions_chart.show()

A horizontal bar chart is used here for the same reason as in Phase 1 - sport names sit more clearly on the y-axis than the x-axis. Sorting from lowest to highest means the most prominent sport is at the top, which is the natural reading direction.

In [19]:
month_pattern = re.compile(r'\[(\d{4}-\d{2})-\d{2}')
monthly_tally = Counter()

for headline in guardian_articles.keys():
    found = month_pattern.search(headline)
    if found:
        monthly_tally[found.group(1)] += 1

monthly_df = pd.DataFrame(monthly_tally.items(), columns=["Month", "ArticleCount"])
monthly_df = monthly_df.sort_values("Month")

coverage_chart = px.bar(
    monthly_df,
    x="Month",
    y="ArticleCount",
    title="Guardian Australia Articles About Brisbane 2032 Sport by Month",
    labels={"ArticleCount": "Number of Articles", "Month": "Month"}
)
coverage_chart.show()

The publication date is extracted from each article's headline key using a regular expression, then grouped by month. This chart helps identify whether the coverage spike seen in the data reflects genuine growing Olympic interest or is driven by unrelated events - which turns out to be an important part of interpreting the findings.

### Insights

**Football dominates, everything else is marginal.** Football/soccer leads with 37 mentions across 50 articles, rugby follows with 14, and athletics with 12. Every other sport has fewer than 10 mentions. This reflects the general structure of Australian sports media rather than any specific Olympic focus - football and rugby dominate the conversation regardless of the 2032 context.

**The mid-tier sports from Phase 1 are almost invisible.** Hockey, identified in Phase 1 as the strongest candidate for ASC investment due to its growing participation trend, registers near zero mentions. Volleyball, Surfing and Taekwondo - all growing in youth participation - are absent entirely. This is the central finding: there is a clear disconnect between where young Australians are actively participating and what is being discussed publicly in the lead-up to 2032.

**The September 2025 spike in the timeline is misleading.** The surge to 16 articles in that month is driven by the AFL grand final between the Brisbane Lions and Geelong, not by any increase in Olympic-focused reporting. This is a limitation of the search approach - the term 'Brisbane' pulls in a large volume of AFL and general Brisbane city content that mentions the Olympics only incidentally. A more targeted search restricted to the Guardian's sport section and using the specific Olympic sport names would produce a cleaner dataset.

**The coverage gap is the ASC's opportunity.** Phase 1 showed that Hockey, Volleyball, Surfing and Taekwondo all have tens of thousands of young participants and positive or stable trends. Phase 2 shows that none of these sports are present in the public conversation around Brisbane 2032. This means the ASC is not competing against existing media attention to reach young fans of these sports - the space is open. A targeted media and social media campaign connecting junior participants in these sports to the 2032 Olympic programme would directly address suggestion 3 from the scenario, reaching young people through topics where they currently have no Olympic connection at all.

This gap between participation and media visibility highlights a critical strategic opportunity. The ASC does not need to create interest in these sports from scratch - it already exists among young participants. Instead, the focus should be on increasing visibility and connection, converting existing participation into long-term fan engagement ahead of Brisbane 2032.